[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Bienvenida, reglas del curso y entorno de trabajo.

**py501 · Aplicaciones Agénticas con LangChain** · Pythonista® · Instructor: José Luis Chiquete · Módulo 1: *Fundamentos de sistemas agénticos*

En este *notebook* se presentan las reglas del curso, se configuran las claves de API como secretos de *GitHub Codespaces* y se verifica que el modelo, *LangSmith* y el servidor MCP del curso respondan.

---
## 1. Bienvenida y reglas del curso.

* El curso dura **16 horas** en 8 sesiones de 2 horas y combina teoría, casos de diseño y laboratorios.
* Los *notebooks* `02` a `14` tienen **ejercicios**; las versiones resueltas están en `soluciones/`. El `15` es la demo final del instructor.
* A lo largo de los laboratorios se construye un mismo sistema: el **asistente de soporte interno** de *Andina Logística*, una empresa ficticia. Consulta políticas (`data/politicas/`), gestiona tickets mediante un servidor MCP (`mcp_servers/tickets_server.py`) y escala casos a una persona.
* Los modelos de lenguaje no son deterministas: tus respuestas no serán idénticas a las del instructor ni a las de tus compañeros. Lo que importa es el **comportamiento**: qué herramientas llama el agente, en qué orden y con qué argumentos.
* Cada llamada consume tokens de **tu** clave de API. Las celdas están pensadas para ejecutarse pocas veces; evita bucles grandes.

| Módulo | Tema | *Notebooks* |
|---|---|---|
| 1 | Fundamentos de sistemas agénticos | `01`-`02` |
| 2 | El ecosistema *LangChain* v1 | `03`-`04` |
| 3 | Herramientas, contexto y RAG | `05`-`06` |
| 4 | MCP y comunicación entre agentes | `07` |
| 5 | *Middleware*: controlar el ciclo del agente | `08` |
| 6 | *LangGraph*: estado, durabilidad y multiagente | `09`-`10` |
| 7 | Memoria y *Deep Agents* | `11`-`12` |
| 8 | Evaluación y observabilidad | `13` |
| 9 | Seguridad, costos y despliegue | `14` |
| 10 | Demo final integradora | `15` |

---
## 2. Claves de API como secretos de *Codespaces*.

Las claves **nunca** se escriben en un *notebook* ni en un archivo del repositorio: este repositorio es público.

1. Abre **github.com/settings/codespaces** → **Codespaces secrets** → **New secret**.
2. Crea `GOOGLE_API_KEY` (obligatoria; se obtiene en *Google AI Studio*) y `LANGSMITH_API_KEY` (recomendada; se obtiene en *smith.langchain.com*). `OPENAI_API_KEY` y `ANTHROPIC_API_KEY` son opcionales.
3. En cada secreto, en **Repository access**, agrega este repositorio.
4. Si el *Codespace* ya estaba abierto, aparecerá un aviso para recargarlo; acéptalo para que las variables existan.

> 💡 Si trabajas en tu máquina, copia `.env.example` a `.env` y completa los valores: `.env` está en `.gitignore`.

---
## 3. Verificación del entorno.

* La siguiente celda mostrará la versión de *Python* y de las bibliotecas del curso. Debe ser *Python* 3.12 y *LangChain* 1.4.

In [ ]:
import sys
from importlib.metadata import version

paquetes = ["langchain", "langchain-core", "langgraph", "langsmith",
            "langchain-google-genai", "langchain-mcp-adapters", "deepagents"]
{"Python": sys.version.split()[0], **{p: version(p) for p in paquetes}}

### Preparación del entorno.

* La siguiente celda localizará `data/`, cargará `.env` si existe (ejecución local), definirá el modelo del curso y verificará la clave de API.

In [ ]:
import os

from dotenv import load_dotenv

RAIZ = next(p for p in (os.curdir, os.pardir)
            if os.path.isdir(os.path.join(p, "data", "politicas")))
load_dotenv(os.path.join(RAIZ, ".env"))   # solo tiene efecto en ejecución local
DATA = os.path.join(RAIZ, "data")
POLITICAS = os.path.join(DATA, "politicas")
MODELO = os.environ.get("MODELO", "google_genai:gemini-3.5-flash")
EMBEDDINGS = os.environ.get("EMBEDDINGS", "google_genai:gemini-embedding-001")

if not os.environ.get("GOOGLE_API_KEY") and MODELO.startswith("google_genai"):
    raise RuntimeError("Falta GOOGLE_API_KEY: agrégala como secreto de Codespaces y reconstruye el Codespace.")
MODELO

* La siguiente celda hará la primera llamada al modelo.

In [ ]:
from langchain.chat_models import init_chat_model

modelo = init_chat_model(MODELO)
modelo.invoke("Responde solo con la palabra: listo").text

* Si configuraste `LANGSMITH_API_KEY`, abre *smith.langchain.com* → proyecto **py501**: la llamada anterior debe aparecer como una traza.

* La siguiente celda arrancará el servidor MCP de tickets y listará sus herramientas. En *Jupyter* se usa `await` directamente porque el *notebook* ya tiene un *event loop* corriendo: **no** uses `asyncio.run()`.

In [ ]:
import sys

from langchain_mcp_adapters.client import MultiServerMCPClient

cliente = MultiServerMCPClient({"tickets": {
    "transport": "stdio", "command": sys.executable,
    "args": [os.path.join(RAIZ, "mcp_servers", "tickets_server.py")]}})
[h.name for h in await cliente.get_tools()]

---
## ✅ Cierre.

**Práctica:** verificar que las tres comprobaciones (versiones, modelo y servidor MCP) funcionen antes de la sesión 1.

**Siguiente *notebook*:** `02_que_es_un_agente.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>